# Stochastic Optimization

This example follows the [PyPSA documentation](https://docs.pypsa.org/latest/user-guide/optimization/stochastic/) on stochastic optimisation, including conditional value at risk (CVaR) optimisation. We follow the canonical example of a single-node wind, solar, battery, hydrogen storage system at 3-hourly resolution for a year, plus a fossil backup generator. There are many other [examples](https://docs.pypsa.org/latest/examples/examples/) on various topics in the documentation, in case you want to explore further beyond the course.

In [1]:
import pypsa

n = pypsa.examples.model_energy()
n.add(
    "Generator",
    "backup_generator",
    carrier="fossil backup",
    bus="electricity",
    p_nom_extendable=True,
    efficiency=0.5,
    marginal_cost=150,
    capital_cost=46000,
)

n.optimize(log_to_console=False)

INFO:pypsa.network.io:Retrieving network data from https://github.com/PyPSA/PyPSA/raw/v1.1.2/examples/networks/model-energy/model-energy.nc.
INFO:pypsa.network.io:Imported network 'Model-Energy' has buses, carriers, generators, links, loads, storage_units, stores
/tmp/ipykernel_982841/3560132789.py:15: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['backup_generator'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 11/11 [00:00<00:00, 578.58it/s]
INFO:linopy.io: Writing time: 0.24s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
S

('ok', 'optimal')

In [2]:
cap_deterministic = n.statistics.optimal_capacity().round(1)
cap_deterministic

component    carrier        
Generator    fossil backup       6271.2
             load shedding      10901.2
             solar              22071.0
             wind               21467.3
StorageUnit  battery storage     9305.5
dtype: float64

In [3]:
obj_deterministic = n.objective / 1e9
obj_deterministic

5.845665755115514

In [4]:
n.set_scenarios({"volcano": 0.2, "no_volcano": 0.8})


In [5]:
n.generators_t.p_max_pu.loc[:, ("volcano", "solar")] *= 0.1
n.generators_t.p_max_pu.loc["2019-07-21"]


scenario            volcano         no_volcano        
name                  solar    wind      solar    wind
snapshot                                              
2019-07-21 00:00:00  0.0000  0.2201      0.000  0.2201
2019-07-21 03:00:00  0.0004  0.1682      0.004  0.1682
2019-07-21 06:00:00  0.0268  0.0715      0.268  0.0715
2019-07-21 09:00:00  0.0263  0.1164      0.263  0.1164
2019-07-21 12:00:00  0.0301  0.1646      0.301  0.1646
2019-07-21 15:00:00  0.0228  0.1181      0.228  0.1181
2019-07-21 18:00:00  0.0004  0.0702      0.004  0.0702
2019-07-21 21:00:00  0.0000  0.0896      0.000  0.0896

In [6]:
n.optimize()


/tmp/ipykernel_982841/2571028986.py:1: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

MultiIndex([(   'volcano', 'backup_generator'),
            ('no_volcano', 'backup_generator')],
           names=['scenario', 'name'])
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 11/11 [00:00<00:00, 569.25it/s]
INFO:linopy.io: Writing time: 0.28s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-11vyik0b has 140174 rows; 64247 cols; 271650 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 3e+00]
  Cost    [9e+01, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+04]
Presolving model
73076 rows, 61403 cols, 201708 nonzeros  0s
Dependent equations search running on 17520 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
67236 rows, 55563 cols, 190028 nonzeros  0s
Presolve reductions: rows 67236(-72938); columns 55563(-8684); nonzeros 190028(-81622) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 5840(3.09983e+07) 0s
      15861     1.8179731819e+09 Pr: 12420(1.17788e+10); Du: 0(5.15549e-08) 5s
      47339     6.0812199200e+09 Pr: 14228(7.26572e+07); Du: 0(1.30089e-14) 11s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 64247 primals, 140174 duals
Objective: 6.24e+09
Solver model: available
Solver message: Optimal



      54679     6.2420598321e+09 Pr: 0(0); Du: 0(2.62378e-09) 12s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-11vyik0b
Model status        : Optimal
Simplex   iterations: 54679
Objective value     :  6.2420598321e+09
P-D objective error :  3.8195497239e-16
HiGHS run time      :         12.18


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


('ok', 'optimal')

In [7]:
n.model


Linopy LP model

Variables:
----------
 * Generator-p_nom (name)
 * Link-p_nom (name)
 * Store-e_nom (name)
 * StorageUnit-p_nom (name)
 * Generator-p (scenario, name, snapshot)
 * Link-p (scenario, name, snapshot)
 * Store-e (scenario, name, snapshot)
 * StorageUnit-p_dispatch (scenario, name, snapshot)
 * StorageUnit-p_store (scenario, name, snapshot)
 * StorageUnit-state_of_charge (scenario, name, snapshot)
 * Store-p (scenario, name, snapshot)

Constraints:
------------
 * Generator-ext-p_nom-lower (name, scenario)
 * Link-ext-p_nom-lower (name, scenario)
 * Store-ext-e_nom-lower (name, scenario)
 * StorageUnit-ext-p_nom-lower (name, scenario)
 * Generator-fix-p-lower (scenario, name, snapshot)
 * Generator-fix-p-upper (scenario, name, snapshot)
 * Generator-ext-p-lower (scenario, name, snapshot)
 * Generator-ext-p-upper (scenario, name, snapshot)
 * Link-ext-p-lower (scenario, name, snapshot)
 * Link-ext-p-upper (scenario, name, snapshot)
 * Store-ext-e-lower (scenario, name, snap

In [8]:
cap_stochastic = n.statistics.optimal_capacity().round(1).unstack(level="scenario")
cap_stochastic


scenario                     no_volcano  volcano
component   carrier                             
Generator   fossil backup        7159.2   7159.2
            load shedding       10901.2  10901.2
            solar               16359.1  16359.1
            wind                25326.7  25326.7
StorageUnit battery storage      5075.2   5075.2

In [9]:
cap_stochastic.iloc[:, 0].div(cap_deterministic).round(2)


component    carrier        
Generator    fossil backup      1.14
             load shedding      1.00
             solar              0.74
             wind               1.18
StorageUnit  battery storage    0.55
dtype: float64

In [10]:
n.objective / 1e9 / obj_deterministic


1.067809911408974

In [11]:
n.statistics.energy_balance().div(1e6).round(2).unstack(level="scenario").xs(
    "electricity", level="bus_carrier"
)


scenario                     no_volcano  volcano
component   carrier                             
Generator   fossil backup         12.15    22.91
            load shedding          0.01     0.04
            solar                 17.01     1.67
            wind                  37.35    41.75
Load        -                    -66.27   -66.27
StorageUnit battery storage       -0.26    -0.10

In [12]:
n.optimize()

n.set_risk_preference(alpha=0.8, omega=0.5)
n.optimize()

n.set_risk_preference(alpha=0.8, omega=0.0)
n.optimize()


/tmp/ipykernel_982841/1886110872.py:1: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

MultiIndex([(   'volcano', 'backup_generator'),
            ('no_volcano', 'backup_generator')],
           names=['scenario', 'name'])
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 11/11 [00:00<00:00, 598.32it/s]
INFO:linopy.io: Writing time: 0.22s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-f9xbhrv3 has 140174 rows; 64247 cols; 271650 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 3e+00]
  Cost    [9e+01, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+04]
Presolving model
73076 rows, 61403 cols, 201708 nonzeros  0s
Dependent equations search running on 17520 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
67236 rows, 55563 cols, 190028 nonzeros  0s
Presolve reductions: rows 67236(-72938); columns 55563(-8684); nonzeros 190028(-81622) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 5840(3.09983e+07) 0s
      15861     1.8179731819e+09 Pr: 12420(1.17788e+10); Du: 0(5.15549e-08) 5s
      47339     6.0812199200e+09 Pr: 14228(7.26572e+07); Du: 0(1.30089e-14) 10s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 64247 primals, 140174 duals
Objective: 6.24e+09
Solver model: available
Solver message: Optimal



      54679     6.2420598321e+09 Pr: 0(0); Du: 0(2.62378e-09) 12s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-f9xbhrv3
Model status        : Optimal
Simplex   iterations: 54679
Objective value     :  6.2420598321e+09
P-D objective error :  3.8195497239e-16
HiGHS run time      :         11.83


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.
/tmp/ipykernel_982841/1886110872.py:4: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

MultiIndex([(   'volcano', 'backup_generator'),
            ('no_volcano', 'backup_generator')],
           names=['scenario', 'name'])
INFO:linopy.

...

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-8hcn9afw has 140177 rows; 64251 cols; 283338 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 6e+03]
  Cost    [9e+01, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+04]
Presolving model
73078 rows, 61406 cols, 213392 nonzeros  0s
Dependent equations search running on 17520 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
67236 rows, 55563 cols, 190028 nonzeros  0s
Presolve reductions: rows 67236(-72941); columns 55563(-8688); nonzeros 190028(-93310) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 5840(3.09983e+07) 0s
      15861     1.8179731819e+09 Pr: 12420(1.17788e+10); Du: 0(5.15549e-08) 5s
      46447     6.0309881010e+09 Pr: 4448(8.70178e+07) 10s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 64251 primals, 140177 duals
Objective: 6.24e+09
Solver model: available
Solver message: Optimal



      54679     6.2420598321e+09 Pr: 0(0); Du: 0(2.62378e-09) 12s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-8hcn9afw
Model status        : Optimal
Simplex   iterations: 54679
Objective value     :  6.2420598321e+09
P-D objective error :  3.8195497239e-16
HiGHS run time      :         12.20


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance, CVaR-excess-volcano, CVaR-excess-no_volcano, CVaR-def were not assigned to the network.


('ok', 'optimal')

In [13]:
p_worst = float(n.scenario_weightings.loc["volcano", "weight"])
n.set_risk_preference(alpha=1 - p_worst, omega=1.0)
n.optimize()


/tmp/ipykernel_982841/332690220.py:3: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

MultiIndex([(   'volcano', 'backup_generator'),
            ('no_volcano', 'backup_generator')],
           names=['scenario', 'name'])
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 14/14 [00:00<00:00, 631.31it/s]
INFO:linopy.io: Writing time: 0.25s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-koq18w60 has 140177 rows; 64251 cols; 283338 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 6e+03]
  Cost    [1e+00, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+04]
Presolving model
73078 rows, 61404 cols, 213390 nonzeros  0s
Dependent equations search running on 17520 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
67238 rows, 55564 cols, 201710 nonzeros  0s
Presolve reductions: rows 67238(-72939); columns 55564(-8687); nonzeros 201710(-81628) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.2800000000e+05 Ph1: 2(2000); Du: 1(128) 0s
      16168     2.9789844015e+09 Pr: 6796(4.5796e+08); Du: 0(1.81662e-07) 6s
      26162     6.7176693055e+09 Pr: 26055(3.50984e+07); Du: 0(3.50908e-08) 11s
      32667     6

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 64251 primals, 140177 duals
Objective: 6.80e+09
Solver model: available
Solver message: Optimal



      36163     6.8024169308e+09 Pr: 0(0); Du: 0(1.1724e-12) 20s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-koq18w60
Model status        : Optimal
Simplex   iterations: 36163
Objective value     :  6.8024169308e+09
P-D objective error :  6.3088376781e-16
HiGHS run time      :         19.94


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance, CVaR-excess-volcano, CVaR-excess-no_volcano, CVaR-def were not assigned to the network.


('ok', 'optimal')

In [14]:
cap_cvar = n.statistics.optimal_capacity().round(1).unstack(level="scenario")
cap_diff = cap_cvar.iloc[:, 0] - cap_stochastic.iloc[:, 0]
print("Capacity diff (omega=0.5 - neutral) [MW]:")
print(cap_diff.round(2))

Capacity diff (omega=0.5 - neutral) [MW]:
component    carrier        
Generator    fossil backup      1503.5
             load shedding         0.0
             solar                 NaN
             wind               5153.3
StorageUnit  battery storage   -4335.8
Name: no_volcano, dtype: float64


In [15]:
n.set_risk_preference(alpha=0.8, omega=0.5)
n.optimize()
obj_cvar = n.objective / 1e9

/tmp/ipykernel_982841/3585241586.py:2: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

MultiIndex([(   'volcano', 'backup_generator'),
            ('no_volcano', 'backup_generator')],
           names=['scenario', 'name'])
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 14/14 [00:00<00:00, 595.27it/s]
INFO:linopy.io: Writing time: 0.26s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-tftq93ze has 140177 rows; 64251 cols; 283338 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 6e+03]
  Cost    [5e-01, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+04]
Presolving model
73078 rows, 61404 cols, 213390 nonzeros  0s
Dependent equations search running on 17520 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
67238 rows, 55564 cols, 201710 nonzeros  0s
Presolve reductions: rows 67238(-72939); columns 55564(-8687); nonzeros 201710(-81628) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -6.4000000000e+04 Ph1: 2(2000); Du: 1(64) 0s
      18480     2.9700421849e+09 Pr: 17950(1.14668e+10); Du: 0(1.41691e-07) 5s
      24368     2.9721949395e+09 Pr: 19177(9.26117e+10); Du: 0(6.32685e-07) 11s
      29148     

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 64251 primals, 140177 duals
Objective: 6.75e+09
Solver model: available
Solver message: Optimal



      50007     6.7482110123e+09 Pr: 0(0); Du: 0(5.0505e-10) 32s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-tftq93ze
Model status        : Optimal
Simplex   iterations: 50007
Objective value     :  6.7482110123e+09
P-D objective error :  2.5438057083e-15
HiGHS run time      :         32.04


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance, CVaR-excess-volcano, CVaR-excess-no_volcano, CVaR-def were not assigned to the network.


In [16]:
n.set_risk_preference(alpha=0.8, omega=0.0)
n.optimize()
obj_risk_neutral = n.objective / 1e9

/tmp/ipykernel_982841/99901509.py:2: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

MultiIndex([(   'volcano', 'backup_generator'),
            ('no_volcano', 'backup_generator')],
           names=['scenario', 'name'])
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 14/14 [00:00<00:00, 596.84it/s]
INFO:linopy.io: Writing time: 0.27s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-z3ioucgl has 140177 rows; 64251 cols; 283338 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 6e+03]
  Cost    [9e+01, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+04]
Presolving model
73078 rows, 61406 cols, 213392 nonzeros  0s
Dependent equations search running on 17520 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
67236 rows, 55563 cols, 190028 nonzeros  0s
Presolve reductions: rows 67236(-72941); columns 55563(-8688); nonzeros 190028(-93310) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 5840(3.09983e+07) 0s
      15311     1.8177716634e+09 Pr: 12236(8.7361e+09); Du: 0(5.15549e-08) 5s
      41512     5.8184655650e+09 Pr: 23964(2.0731e+08) 11s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 64251 primals, 140177 duals
Objective: 6.24e+09
Solver model: available
Solver message: Optimal



      54679     6.2420598321e+09 Pr: 0(0); Du: 0(2.62378e-09) 14s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-z3ioucgl
Model status        : Optimal
Simplex   iterations: 54679
Objective value     :  6.2420598321e+09
P-D objective error :  3.8195497239e-16
HiGHS run time      :         13.62


INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance, CVaR-excess-volcano, CVaR-excess-no_volcano, CVaR-def were not assigned to the network.


In [17]:
premium = obj_cvar - obj_risk_neutral
print(
    f"Insurance Premium: {premium:.4f} billion EUR ({premium / obj_risk_neutral * 100:.2f}%)"
)

Insurance Premium: 0.5062 billion EUR (8.11%)


## Exercises

**Task 1:** Replace the time series for wind and solar with the time series for the Philippines (https://model.energy/data/time-series-fe86a41f1102dcdc259428e80b379b42.csv) and re-run the stochastic optimisation. How do the optimal capacities and insurance premium change compared to the original case?